In [1]:
!pip install yfinance


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import yfinance as yf
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, mean_absolute_error

In [3]:
# Last 6 months of data
df = yf.download("AAPL", period="6mo", interval="1d")

if isinstance(df.columns, pd.MultiIndex):
    df.columns = [col[0] for col in df.columns]

df = df.reset_index()
df = df.sort_values("Date")

[*********************100%***********************]  1 of 1 completed


In [4]:
# --- Feature Engineering ---

df["sma_10"] = df["Close"].rolling(10).mean()
df["sma_20"] = df["Close"].rolling(20).mean()

delta = df["Close"].diff()
gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)
avg_gain = gain.rolling(14).mean()
avg_loss = loss.rolling(14).mean()
rs = avg_gain / avg_loss
df["rsi_14"] = 100 - (100 / (1 + rs))

ema12 = df["Close"].ewm(span=12, adjust=False).mean()
ema26 = df["Close"].ewm(span=26, adjust=False).mean()
df["macd"] = ema12 - ema26
df["macd_signal"] = df["macd"].ewm(span=9, adjust=False).mean()

rolling_mean = df["Close"].rolling(20).mean()
rolling_std  = df["Close"].rolling(20).std()
df["bb_upper"] = rolling_mean + (2 * rolling_std)
df["bb_lower"] = rolling_mean - (2 * rolling_std)
df["bb_width"] = (df["bb_upper"] - df["bb_lower"]) / df["Close"]

df["return"]     = df["Close"].pct_change()
df["return_3d"]  = df["Close"].pct_change(3)
df["return_5d"]  = df["Close"].pct_change(5)

df["volatility_5d"]  = df["return"].rolling(5).std()
df["volatility_10d"] = df["return"].rolling(10).std()

df["dist_sma_10"] = (df["Close"] - df["sma_10"]) / df["sma_10"]
df["dist_sma_20"] = (df["Close"] - df["sma_20"]) / df["sma_20"]

# Targets
df["target_signal"] = (df["return"].shift(-1) > 0.002).astype(int)
df["target_price"]  = df["Close"].shift(-1)

df = df.dropna()

In [5]:
feature_cols = [
    "rsi_14", "macd", "macd_signal", "bb_width",
    "return_3d", "return_5d", "volatility_5d", "volatility_10d",
    "dist_sma_10", "dist_sma_20", "Volume"
]

X        = df[feature_cols]
y_signal = df["target_signal"]
y_price  = df["target_price"]

split_index = int(len(df) * 0.8)

X_train, X_test         = X[:split_index], X[split_index:]
y_sig_train, y_sig_test = y_signal[:split_index], y_signal[split_index:]
y_prc_train, y_prc_test = y_price[:split_index], y_price[split_index:]

In [6]:
# --- Classifier: BUY / SELL / HOLD signal ---
# No scaling needed — HistGBM handles it internally via histogram binning
hgb_clf = HistGradientBoostingClassifier(
    max_iter=500,
    max_depth=4,
    learning_rate=0.05,
    min_samples_leaf=10,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=20,
    random_state=42
)
hgb_clf.fit(X_train, y_sig_train)

clf_pred = hgb_clf.predict(X_test)
print("HistGBM Classifier Accuracy:", accuracy_score(y_sig_test, clf_pred))
print(classification_report(y_sig_test, clf_pred))
print("ROC-AUC:", roc_auc_score(y_sig_test, hgb_clf.predict_proba(X_test)[:, 1]))

HistGBM Classifier Accuracy: 0.42857142857142855
              precision    recall  f1-score   support

           0       1.00      0.08      0.14        13
           1       0.40      1.00      0.57         8

    accuracy                           0.43        21
   macro avg       0.70      0.54      0.36        21
weighted avg       0.77      0.43      0.31        21

ROC-AUC: 0.6298076923076923


In [7]:
# --- Regressor: Next day closing price ---
hgb_reg = HistGradientBoostingRegressor(
    max_iter=500,
    max_depth=4,
    learning_rate=0.05,
    min_samples_leaf=10,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=20,
    random_state=42
)
hgb_reg.fit(X_train, y_prc_train)

price_pred = hgb_reg.predict(X_test)
print("HistGBM Regressor MAE: $", round(mean_absolute_error(y_prc_test, price_pred), 2))

HistGBM Regressor MAE: $ 7.15


In [8]:
# --- Combined output table ---
results = X_test.copy()
results["actual_next_close"]    = y_prc_test.values
results["predicted_next_close"] = price_pred
results["actual_signal"]        = y_sig_test.values
results["predicted_signal"]     = clf_pred
print(results[["actual_next_close", "predicted_next_close", "actual_signal", "predicted_signal"]].tail(10))

     actual_next_close  predicted_next_close  actual_signal  predicted_signal
114         264.720001            253.973811              1                 1
115         263.750000            254.807829              0                 1
116         262.519989            251.891410              0                 1
117         260.290009            250.194791              0                 1
118         257.459991            254.699166              0                 1
119         259.880005            261.859732              1                 1
120         260.829987            257.655123              1                 1
121         260.809998            263.123695              0                 1
122         255.759995            272.024702              0                 1
123         250.119995            261.203275              0                 1


In [9]:
# --- Latest signal + next day price forecast ---
latest = X_test.iloc[[-1]]

prob            = hgb_clf.predict_proba(latest)[:, 1][0]
predicted_price = hgb_reg.predict(latest)[0]

if prob > 0.6:
    signal = "BUY"
elif prob < 0.4:
    signal = "SELL"
else:
    signal = "HOLD"

print(f"Signal:               {signal}")
print(f"Confidence:           {round(prob, 2)}")
print(f"Predicted Next Close: ${round(predicted_price, 2)}")

Signal:               BUY
Confidence:           0.86
Predicted Next Close: $261.2
